In [1]:
%pwd

'a:\\projects\\text-summarizer\\research'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'a:\\projects\\text-summarizer'

# Entity

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: Path
    ALL_REQUIRED_FILES: list

# Configuration manager

In [9]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from pathlib import Path

In [10]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories(path_to_directories=[Path(self.config.artifacts_root)])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        root_dir = Path(config.root_dir)
        status_file = Path(config.STATUS_FILE)

        create_directories(path_to_directories=[root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=root_dir,
            STATUS_FILE=status_file,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES
        )

        return data_validation_config

# Components

In [11]:
import os

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exist(self) -> bool:
        try:
            validation_status = True 
            data_ingestion_dir = os.path.join("artifacts", "data_ingestion")
            all_files = os.listdir(data_ingestion_dir)

            for file in self.config.ALL_REQUIRED_FILES:
                if file not in all_files:
                    validation_status = False
                    break  

            os.makedirs(os.path.dirname(self.config.STATUS_FILE), exist_ok=True)

            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e

# Pipeline

In [13]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files_exist()
except Exception as e:
    raise e

[2026-06-19 11:35:11,429: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-19 11:35:11,431: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-19 11:35:11,432: INFO: common: Created directory at: artifacts]
[2026-06-19 11:35:11,434: INFO: common: Created directory at: artifacts\data_validation]
